In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/rockyt07/data-center-dataset/Book1 (1).csv
/kaggle/input/datasets/rockyt07/data-center-dataset/Book1.xlsx
/kaggle/input/datasets/claytonmiller/cubems-smart-building-energy-and-iaq-data/2018Floor1.csv
/kaggle/input/datasets/claytonmiller/cubems-smart-building-energy-and-iaq-data/2018Floor4.csv
/kaggle/input/datasets/claytonmiller/cubems-smart-building-energy-and-iaq-data/2019Floor6.csv
/kaggle/input/datasets/claytonmiller/cubems-smart-building-energy-and-iaq-data/2018Floor6.csv
/kaggle/input/datasets/claytonmiller/cubems-smart-building-energy-and-iaq-data/2019Floor1.csv
/kaggle/input/datasets/claytonmiller/cubems-smart-building-energy-and-iaq-data/2019Floor2.csv
/kaggle/input/datasets/claytonmiller/cubems-smart-building-energy-and-iaq-data/2018Floor5.csv
/kaggle/input/datasets/claytonmiller/cubems-smart-building-energy-and-iaq-data/2019Floor3.csv
/kaggle/input/datasets/claytonmiller/cubems-smart-building-energy-and-iaq-data/2019Floor7.csv
/kaggle/input/datasets/cl

# Code for normalizing all the csv's in the CU-BEMS dataset into a single csv

In [29]:
import pandas as pd
import os
import re

folder_path = "/kaggle/input/datasets/claytonmiller/cubems-smart-building-energy-and-iaq-data"

dfs = []

for file in os.listdir(folder_path):
    if file.endswith(".csv"):
        file_path = os.path.join(folder_path, file)
        
        df = pd.read_csv(file_path)
        
        # Clean column names
        df.columns = df.columns.str.strip().str.lower()
        
        # Extract metadata
        match = re.match(r"(\d{4})Floor(\d+)", file)
        if not match:
            continue
        
        year = int(match.group(1))
        floor = int(match.group(2))
        
        # Add metadata
        df["year"] = year
        df["floor"] = floor
        
        # ✅ Robust timestamp handling (FIXED)
        if "date" in df.columns:
            df["date"] = pd.to_datetime(df["date"], errors='coerce')
            df = df.dropna(subset=["date"])
        elif "timestamp" in df.columns:
            df["timestamp"] = pd.to_datetime(df["timestamp"], errors='coerce')
            df = df.dropna(subset=["timestamp"])
        
        dfs.append(df)

# Combine all dataframes
final_df = pd.concat(dfs, ignore_index=True, sort=False)

# Handle missing values
final_df = final_df.fillna(0)

# ✅ Sorting
if "date" in final_df.columns:
    final_df = final_df.sort_values(by=["year", "floor", "date"])
elif "timestamp" in final_df.columns:
    final_df = final_df.sort_values(by=["year", "floor", "timestamp"])

# Reset index
final_df = final_df.reset_index(drop=True)

# ✅ Move important columns to front
priority_cols = ["year", "floor"]

if "date" in final_df.columns:
    priority_cols.append("date")
elif "timestamp" in final_df.columns:
    priority_cols.append("timestamp")

cols = priority_cols + [col for col in final_df.columns if col not in priority_cols]
final_df = final_df[cols]

# Save final dataset
final_df.to_csv("combined_dataset.csv", index=False)

print("✅ Done!", final_df.shape)

✅ Done! (5432488, 51)


In [30]:
cols = ['year', 'floor'] + [col for col in final_df.columns if col not in ['year', 'floor']]
final_df = final_df[cols]

In [31]:
final_df.to_parquet("combined_dataset.parquet")

In [32]:
df = pd.read_parquet("/kaggle/working/combined_dataset.parquet")

In [24]:
len(df)

5432489

In [33]:
df.head()

,year,floor,date,z1_light(kw),z1_plug(kw),z2_ac1(kw),z2_ac2(kw),z2_ac3(kw),z2_ac4(kw),z2_light(kw),...,z2_ac8(kw),z2_ac9(kw),z2_ac10(kw),z2_ac11(kw),z2_ac12(kw),z2_ac13(kw),z2_ac14(kw),z3_s1(degc),z3_s1(rh%),z3_s1(lux)
0,2018,1,2018-07-01 00:00:00,12.94,18.56,45.24,0.01,0.01,0.00,13.76,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2018,1,2018-07-01 00:01:00,12.97,18.55,45.28,0.02,0.01,0.01,13.76,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2018,1,2018-07-01 00:02:00,12.97,18.55,45.24,0.01,0.01,0.01,13.79,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2018,1,2018-07-01 00:03:00,12.98,18.58,45.26,0.02,0.01,0.00,13.81,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2018,1,2018-07-01 00:04:00,13.01,18.60,45.22,0.02,0.01,0.01,13.83,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [34]:
df['year'].value_counts()

year
2019    3577768
2018    1854720
Name: count, dtype: int64

In [35]:
df['year'].unique

<bound method Series.unique of 0          2018
1          2018
2          2018
3          2018
4          2018
           ... 
5432483    2019
5432484    2019
5432485    2019
5432486    2019
5432487    2019
Name: year, Length: 5432488, dtype: int64>

In [16]:
df.columns

Index(['year', 'floor', 'date', 'z1_light(kw)', 'z1_plug(kw)', 'z2_ac1(kw)',
       'z2_ac2(kw)', 'z2_ac3(kw)', 'z2_ac4(kw)', 'z2_light(kw)', 'z2_plug(kw)',
       'z3_light(kw)', 'z3_plug(kw)', 'z4_light(kw)', 'z1_ac1(kw)',
       'z1_ac2(kw)', 'z1_ac3(kw)', 'z1_ac4(kw)', 'z1_s1(degc)', 'z1_s1(rh%)',
       'z1_s1(lux)', 'z2_s1(degc)', 'z2_s1(rh%)', 'z2_s1(lux)', 'z4_ac1(kw)',
       'z4_plug(kw)', 'z4_s1(degc)', 'z4_s1(rh%)', 'z4_s1(lux)', 'z5_ac1(kw)',
       'z5_light(kw)', 'z5_plug(kw)', 'z5_s1(degc)', 'z5_s1(rh%)',
       'z5_s1(lux)', 'z4_ac2(kw)', 'z4_ac3(kw)', 'z4_ac4(kw)', 'z2_ac5(kw)',
       'z2_ac6(kw)', 'z2_ac7(kw)', 'z2_ac8(kw)', 'z2_ac9(kw)', 'z2_ac10(kw)',
       'z2_ac11(kw)', 'z2_ac12(kw)', 'z2_ac13(kw)', 'z2_ac14(kw)',
       'z3_s1(degc)', 'z3_s1(rh%)', 'z3_s1(lux)'],
      dtype='object')

In [17]:
df.isnull().sum()

year            0
floor           0
date            0
z1_light(kw)    0
z1_plug(kw)     0
z2_ac1(kw)      0
z2_ac2(kw)      0
z2_ac3(kw)      0
z2_ac4(kw)      0
z2_light(kw)    0
z2_plug(kw)     0
z3_light(kw)    0
z3_plug(kw)     0
z4_light(kw)    0
z1_ac1(kw)      0
z1_ac2(kw)      0
z1_ac3(kw)      0
z1_ac4(kw)      0
z1_s1(degc)     0
z1_s1(rh%)      0
z1_s1(lux)      0
z2_s1(degc)     0
z2_s1(rh%)      0
z2_s1(lux)      0
z4_ac1(kw)      0
z4_plug(kw)     0
z4_s1(degc)     0
z4_s1(rh%)      0
z4_s1(lux)      0
z5_ac1(kw)      0
z5_light(kw)    0
z5_plug(kw)     0
z5_s1(degc)     0
z5_s1(rh%)      0
z5_s1(lux)      0
z4_ac2(kw)      0
z4_ac3(kw)      0
z4_ac4(kw)      0
z2_ac5(kw)      0
z2_ac6(kw)      0
z2_ac7(kw)      0
z2_ac8(kw)      0
z2_ac9(kw)      0
z2_ac10(kw)     0
z2_ac11(kw)     0
z2_ac12(kw)     0
z2_ac13(kw)     0
z2_ac14(kw)     0
z3_s1(degc)     0
z3_s1(rh%)      0
z3_s1(lux)      0
dtype: int64

In [ ]:
df_gdcd=pd.read_csv("/kaggle/input/datasets/rockyt07/data-center-dataset/Book1 (1).csv")
df_gdcd.head(5)

In [ ]:
df_gdcd[df_gdcd['country']=='India']

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sc

In [ ]:
df_gdcd.columns

In [ ]:
df_gdcd.isnull().sum()

In [ ]:
len(df_gdcd)

In [ ]:
# df_gdcd['hyperscale_data_centers'] = df_gdcd['hyperscale_data_centers'].astype(str).str.replace(r'\D+', '', regex=True)              #to convert the string 300+ and so on into only a numbered string
# df_gdcd['hyperscale_data_centers'] = df_gdcd['hyperscale_data_centers'].astype(int)   # to convert the string dataype column into integer

In [ ]:
# df_gdcd["power_capacity_MW_total"] = (
#     df_gdcd["power_capacity_MW_total"]
#     .str.replace(r"[~, +]", "", regex=True)
#     .str.replace(",", "")
# )

# # Convert safely (invalid → NaN instead of error)
# df_gdcd["power_capacity_MW_total"] = pd.to_numeric(
#     df_gdcd["power_capacity_MW_total"], errors="coerce"
# )


In [ ]:
cols=['colocation_data_centers','floor_space_sqft_total','power_capacity_MW_total','average_renewable_energy_usage_percent','number_of_fiber_connections','growth_rate_of_data_centers_percent_per_year','hyperscale_data_centers']
for col in cols:
    df_gdcd[col] = (
    df_gdcd[col]
    .str.replace(r"[~, +,%,<,>]", "", regex=True)
    .str.replace(",", "")
)

# Convert safely (invalid → NaN instead of error)
df_gdcd[col] = pd.to_numeric(
    df_gdcd[col], errors="coerce"
)

In [ ]:
df_gdcd

In [ ]:
df_gdcd['power_capacity_MW_total'].dtype

In [ ]:
df_gdcd['country']=df_gdcd['country'].astype("string")
df_gdcd['colocation_data_centers']=df_gdcd['colocation_data_centers'].astype("int64")
df_gdcd['floor_space_sqft_total']=df_gdcd['floor_space_sqft_total'].astype("int64")
# df_gdcd['average_renewable_energy_usage_percent']=df_gdcd['average_renewable_energy_usage_percent'].astype("int64")
# df_gdcd['internet_penetration_percent']=df_gdcd['internet_penetration_percent'].astype("int64")
# df_gdcd['avg_latency_to_global_hubs_ms']=df_gdcd['avg_latency_to_global_hubs_ms'].astype("int64")

In [ ]:
for col in df_gdcd.columns:
    print(col," type: ", df_gdcd[col].dtype,"\n")

In [ ]:
df_sort=df_gdcd.sort_values('total_data_centers', ascending=False).iloc[:10]
df_sort.plot.bar(x='country', y='total_data_centers')

In [ ]:
df_sort=df_gdcd.sort_values('hyperscale_data_centers', ascending=False).iloc[:10]
df_sort.plot.bar(x='country', y='hyperscale_data_centers')

In [ ]:
import matplotlib.pyplot as plt
import math

# 1. Group the data by all three categories
grouped = df.groupby(['Year', 'Region', 'Model'])

# 2. Determine grid size (number of rows/cols)
num_plots = len(grouped)
cols = 3  # Adjust this to make the grid wider or thinner
rows = math.ceil(num_plots / cols)

# 3. Create a large figure to hold all subplots
fig, axes = plt.subplots(rows, cols, figsize=(15, rows * 4))
axes = axes.flatten() # Flatten so we can iterate easily with a single index

# 4. Loop through the groups and plot
for i, ((year, region, model), group) in enumerate(grouped):
    ax = axes[i]
    
    # Sort by Month to ensure the line doesn't jump around
    group = group.sort_values('Month')
    
    ax.plot(group['Month'], group['Units_Sold'], marker='o', color='teal')
    
    # Customizing each subplot
    ax.set_title(f"{year} | {region} | {model}", fontsize=10)
    ax.set_xlabel('Month')
    ax.set_ylabel('Units Sold')
    ax.grid(True, linestyle='--', alpha=0.6)

# 5. Clean up: Remove any empty subplots at the end of the grid
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import interact, widgets

# --- 1. Setup the Update Function ---
def update_plot(region, model, year):
    # Filter the data based on dropdown selections
    filtered = df[(df['Region'] == region) & 
                  (df['Model'] == model) & 
                  (df['Year'] == year)]
    
    # Sort by month to ensure the line flows correctly
    filtered = filtered.sort_values('Month')

    # Clear the previous plot and create a new one
    plt.figure(figsize=(10, 5))
    
    if not filtered.empty:
        plt.plot(filtered['Month'], filtered['Units_Sold'], marker='o', color='tab:blue', linewidth=2)
        plt.title(f'Sales for {model} in {region} ({year})', fontsize=14)
        plt.xlabel('Month')
        plt.ylabel('Units Sold')
        plt.grid(True, linestyle='--', alpha=0.7)
        plt.xticks(range(1, 13)) # Assuming Months are 1-12
    else:
        plt.text(0.5, 0.5, 'No data available for this selection', 
                 horizontalalignment='center', verticalalignment='center')
        plt.title('No Data')
        
    plt.show()

# --- 2. Create the Widgets ---
# We use unique values from your columns to populate the dropdowns
region_dropdown = widgets.Dropdown(options=sorted(df['Region'].unique()), description='Region:')
model_dropdown = widgets.Dropdown(options=sorted(df['Model'].unique()), description='Model:')
year_dropdown = widgets.Dropdown(options=sorted(df['Year'].unique()), description='Year:')

# --- 3. Link Widgets to the Function ---
interact(update_plot, region=region_dropdown, model=model_dropdown, year=year_dropdown);

In [ ]:
df1=df.groupby(['Region','Model'])['Units_Sold']
df1.head()